# 어선 항적 분석 최종 모델
**최종 패키지: 2026-11-01**

같은 위치의 `final_model_assets.zip`과 함께 실행합니다. 대회 원본 외의 외부 데이터셋은 사용하지 않았습니다.
ZIP에는 대회 데이터에서 만든 특징·검증 자료, 동결된 분류 모델과 별도 재학습 이상모델이 들어 있습니다.

기본 실행은 **과제 1 동결 모델 추론 → 과제 2 확정 점수 재현 → 기존 holdout 재평가 → 새 B 입력 추론 예제**입니다.
과제 2의 확정 점수 재현과 새 모델 추론 결과는 서로 다른 파일에 저장합니다.
원래 Test B 고정 모델은 전달되지 않았으므로, 이를 복원했다고 주장하지 않습니다.


## 1. 실행 설정
Google Colab에서는 왼쪽 파일 패널 또는 Drive에 IPYNB와 ZIP을 두고 `BASE_DIR`을 해당 폴더로 지정합니다.
로컬 Jupyter에서는 두 파일을 같은 폴더에 둡니다. 아래부터 순서대로 실행합니다.
기본값으로 원래 확정 결과를 다시 만들 수 있습니다. GPU는 필요하지 않습니다.


In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib.util, zipfile, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo

BASE_DIR = Path.cwd()  # Colab Drive 사용 시 Path('/content/drive/MyDrive/본인폴더')
ASSET_ZIP = BASE_DIR / 'final_model_assets.zip'
ASSETS_DIR = BASE_DIR / 'final_model_assets'
RUN_HOLDOUT_EVALUATION = True  # 1,477척·801,973시간대. CPU에서 수 분 걸릴 수 있음
RUN_NEW_B_EXAMPLE = True       # 별도 재학습 모델의 신규 입력 추론 예제
THREADS = 2
# 새 입력은 추출 완료된 특징 CSV/Parquet를 지정. None이면 ZIP의 확정 입력 사용.
CUSTOM_TASK_A_FEATURES = None
CUSTOM_TASK_B_FEATURES50 = None
NEW_B_TARGET_START = '2026-06-01 00:00:00'
# 새 B의 과거 시점 적용은 기본적으로 차단. 기존 확정 B 재현에는 영향을 주지 않음.
ALLOW_NEW_B_BACKCAST = False
RUN_ID = datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = BASE_DIR / 'outputs' / ('final_run_' + RUN_ID)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print('결과 폴더:', OUTPUT_DIR)


결과 폴더: /private/tmp/kisti_final_notebook_c12b003m/outputs/final_run_20261101_153354_473653


## 2. 의존성 준비
검증한 버전을 별도 Python 프로세스에서 확인합니다. 필요하면 설치합니다.
모델 실행은 새 프로세스에서 수행하므로, 설치 전 커널에 남은 라이브러리와 섞이지 않습니다.
Mac에서는 scikit-learn에 포함된 OpenMP 경로를 작업 프로세스에 전달합니다.


In [2]:
REQUIREMENTS = {
    'numpy': '2.4.6', 'pandas': '2.2.3', 'pyarrow': '25.0.1',
    'lightgbm': '4.6.0', 'scikit-learn': '1.7.2', 'joblib': '1.6.0', 'scipy': '1.18.1'
}
if sys.version_info < (3, 11):
    raise RuntimeError('Python 3.11 이상 환경을 사용하세요.')

def worker_environment():
    env = os.environ.copy()
    env['OMP_NUM_THREADS'] = str(THREADS)
    env['OPENBLAS_NUM_THREADS'] = str(THREADS)
    if sys.platform == 'darwin':
        spec = importlib.util.find_spec('sklearn')
        if spec and spec.origin:
            folder = str(Path(spec.origin).parent / '.dylibs')
            env['DYLD_LIBRARY_PATH'] = folder + os.pathsep + env.get('DYLD_LIBRARY_PATH', '')
    return env

PROBE = "import json,numpy,pandas,pyarrow,lightgbm,sklearn,joblib,scipy; print(json.dumps({'numpy':numpy.__version__,'pandas':pandas.__version__,'pyarrow':pyarrow.__version__,'lightgbm':lightgbm.__version__,'scikit-learn':sklearn.__version__,'joblib':joblib.__version__,'scipy':scipy.__version__}))"
probe = subprocess.run([sys.executable, '-c', PROBE], env=worker_environment(), capture_output=True, text=True)
versions = json.loads(probe.stdout.strip().splitlines()[-1]) if probe.returncode == 0 else {}
if any(versions.get(k) != v for k, v in REQUIREMENTS.items()):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--disable-pip-version-check'] +
                          [f'{k}=={v}' for k, v in REQUIREMENTS.items()])
    importlib.invalidate_caches()
    probe = subprocess.run([sys.executable, '-c', PROBE], env=worker_environment(), capture_output=True, text=True)
    if probe.returncode != 0:
        raise RuntimeError(probe.stderr + '\nMac에서 OpenMP가 없으면 libomp 설치가 필요할 수 있습니다.')
    versions = json.loads(probe.stdout.strip().splitlines()[-1])
if any(versions.get(k) != v for k, v in REQUIREMENTS.items()):
    raise RuntimeError('라이브러리 버전이 설정과 다릅니다: ' + str(versions))
print(json.dumps(versions, ensure_ascii=False, indent=2))


{
  "numpy": "2.4.6",
  "pandas": "2.2.3",
  "pyarrow": "25.0.1",
  "lightgbm": "4.6.0",
  "scikit-learn": "1.7.2",
  "joblib": "1.6.0",
  "scipy": "1.18.1"
}


## 3. 추가 자료 압축 해제와 해시 확인
원본 대회 CSV를 다시 준비하지 않아도 포함된 파생 특징으로 최종 실행을 재현할 수 있습니다.
원시 `geom` WKT를 50개 특징으로 바꾸는 기존 코드는 전달 자료에 없어 포함되지 않았습니다.
이 노트북의 새 데이터 입력 경계는 특징 테이블입니다. 파일별 출처와 SHA-256은 ZIP의 manifest에 있습니다.


In [3]:
if not ASSET_ZIP.is_file():
    raise FileNotFoundError('final_model_assets.zip을 BASE_DIR에 두거나 ASSET_ZIP 경로를 바꾸세요.')
with zipfile.ZipFile(ASSET_ZIP) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('ZIP에 중복 경로가 있습니다.')
    for name in names:
        if Path(name).is_absolute() or '..' in Path(name).parts:
            raise ValueError('ZIP 경로가 올바르지 않습니다.')
    archive.extractall(ASSETS_DIR)
asset_manifest = json.loads((ASSETS_DIR / 'manifest.json').read_text(encoding='utf-8'))
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8*1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
for relative, entry in asset_manifest['files'].items():
    if file_sha(ASSETS_DIR / relative) != entry['sha256']:
        raise ValueError('무결성 검사 실패: ' + relative)
print('자료 파일', len(asset_manifest['files']), '개 해시 일치')
print('외부 데이터셋 사용:', asset_manifest['external_datasets_used'])

def run_job(mode, input_path, **options):
    config = dict(mode=mode, assets=str(ASSETS_DIR.resolve()), output=str(OUTPUT_DIR.resolve()),
                  input=str(Path(input_path).resolve()), threads=THREADS, **options)
    job_path = OUTPUT_DIR / (mode + '_job.json')
    job_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
    command = [sys.executable, '-u', str(ASSETS_DIR / 'runtime/final_pipeline.py'), '--job', str(job_path)]
    completed = subprocess.run(command, env=worker_environment(), capture_output=True, text=True)
    (OUTPUT_DIR / (mode + '.log')).write_text(completed.stdout + completed.stderr, encoding='utf-8')
    if completed.returncode:
        raise RuntimeError(completed.stderr or completed.stdout)
    print(completed.stdout)
    return json.loads((OUTPUT_DIR / (mode + '_summary.json')).read_text(encoding='utf-8'))


자료 파일 29 개 해시 일치
외부 데이터셋 사용: False


## 4. 과제 1 — 동결된 분류 모델 추론
LightGBM 3개를 온도 1.0·1.3·1.0으로 보정하고 확률을 동등 평균합니다.
각 선박의 관측 시간대도 동등 평균해 최종 조업형태를 정합니다. ID·정답은 입력 특징에 넣지 않습니다.
129개 특징이 모두 있으면 순서를 검사해 사용합니다. 50개 기본 특징만 제공하면 전체 선박 관측에서 과거 6·12·24시간 요약을 만듭니다.
과거 관측이 입력에서 빠지면 과거 요약과 예측도 달라집니다.

기존 확정 확률을 동일하게 재현할 때는 ZIP의 129개 완성 특징을 사용합니다. 50개 입력에서 과거 요약을 재계산하면 반올림·트리 분기로 일부 시간대 확률이 달라질 수 있습니다. 추가 검사에서 최종 100척 라벨은 같았지만 시간대 확률 최대 차이는 0.0317335였습니다.


In [4]:
task_a_input = Path(CUSTOM_TASK_A_FEATURES) if CUSTOM_TASK_A_FEATURES else ASSETS_DIR / 'inputs/test_a_features129.parquet'
a_summary = run_job('predict_a', task_a_input, check_final=CUSTOM_TASK_A_FEATURES is None)


{
  "mode": "predict_a",
  "hourly_rows": 58189,
  "vessels": 100,
  "prediction_counts": {
    "연안복합": 58,
    "자망": 29,
    "통발": 8,
    "채낚기연승낚시": 5
  },
  "official_score": null,
  "input_sha256": "901be880a8f55b4162a52e907fad6bc88a202765f2defc6efb5b4dfe66bf434d",
  "model_origin": "original_selected_fold_models",
  "final_reference_matches": true,
  "probability_max_error": 1.6653345369377348e-16
}



## 5. 과제 2 — 확정된 Test B 결과 재현
제공된 4개 성분의 가용값 평균으로 확정 점수를 재구성합니다. 원래 Test B 모델의 raw 추론을 새로 수행하는 단계는 아닙니다.
2025년 항적에 2026년 기준을 적용한 사후 점수이며, 실제 사고 확률·조기 탐지 성능으로 해석하지 않습니다.
최고점 시각은 `peak_time`으로 저장합니다. 확인된 사고시각이라는 이름을 붙이지 않습니다.


In [5]:
b_summary = run_job('frozen_b', ASSETS_DIR / 'inputs/test_b_final_components.parquet')


{
  "mode": "frozen_b",
  "hourly_rows": 82626,
  "vessels": 100,
  "low_support_rows": 921,
  "official_score": null,
  "raw_model_inference": false,
  "origin": "supplied_final_test_b_components",
  "as_of_causal": false,
  "input_sha256": "5fb84d55ba4788b0ea77c1843b4e163e20c35f13fa76534ed329c152266dc9f8",
  "note": "2025년 대상에 2026년 기준을 적용한 사후 점수. 최고점 시각은 실제 사고시각이 아닙니다."
}



## 6. 과제 1 — 전체 holdout 점수 재계산
학습에 사용하지 않은 기존 holdout 1,477척·801,973시간대입니다.
선박 Macro F1 약 54.33%, 선박 정확도 약 56.87%, 선박 균형 시간대 Macro F1 약 51.62%를 재현해야 합니다.
과거 단계에서 이미 관찰한 holdout이므로 새로운 독립 시험이나 공식 Test 성적은 아닙니다.


In [6]:
holdout_summary = None
if RUN_HOLDOUT_EVALUATION:
    holdout_summary = run_job('evaluate', ASSETS_DIR / 'evaluation/holdout_features129.parquet')
else:
    print('전체 holdout 재평가를 건너뛰었습니다. 저장 지표를 새로 실행한 성적으로 표시하지 않습니다.')


{
  "rows": 801973,
  "vessels": 1477,
  "vessel_aggregate_macro_f1": 0.5432523244581186,
  "vessel_aggregate_accuracy": 0.5687203791469194,
  "vessel_balanced_hourly_macro_f1": 0.5161555057644679,
  "vessel_balanced_log_loss": 0.9484960295993684,
  "per_class": {
    "자망": {
      "precision": 0.5452196382428941,
      "recall": 0.41865079365079366,
      "f1": 0.47362514029180697
    },
    "통발": {
      "precision": 0.4634146341463415,
      "recall": 0.2714285714285714,
      "f1": 0.34234234234234234
    },
    "연안복합": {
      "precision": 0.5929978118161926,
      "recall": 0.7424657534246575,
      "f1": 0.6593673965936739
    },
    "채낚기연승낚시": {
      "precision": 0.5660377358490566,
      "recall": 0.9090909090909091,
      "f1": 0.6976744186046512
    }
  },
  "scope": "기존 holdout 재검증. 공식 Test 성적이나 새 독립 시험이 아님",
  "expected_metrics_match": true,
  "official_score": null
}



## 7. 과제 2 — 별도 재학습 모델로 새 특징 입력 추론
이 단계는 2026-11-01에 대회 자료로 다시 학습한 June 2026 IsolationForest를 사용합니다.
기본 예제는 3척의 2026년 5~6월 특징입니다. 5월 관측은 개인 과거 이력으로 사용하고 6월부터 점수를 냅니다.
720시간 개인 기준·공간 이탈·직전 변화·peer 성분을 과거 보정 CDF로 바꿔 평균합니다.
CDF는 소수 12자리 동점 규칙을 사용하며 같은 규칙으로 보정 임계값도 다시 계산합니다.
원래 확정된 Test B 점수는 교체하지 않습니다. 모델 적합 시점보다 이른 대상은 기본 차단합니다.


In [7]:
new_b_summary = None
if CUSTOM_TASK_B_FEATURES50:
    new_b_summary = run_job('new_b', Path(CUSTOM_TASK_B_FEATURES50),
                            target_start=NEW_B_TARGET_START, allow_backcast=ALLOW_NEW_B_BACKCAST)
elif RUN_NEW_B_EXAMPLE:
    new_b_summary = run_job('new_b_demo', ASSETS_DIR / 'examples/task_b_features50.parquet',
                            target_start='2026-06-01 00:00:00', allow_backcast=False)
else:
    print('새 B 입력 추론을 실행하지 않았습니다.')


{
  "mode": "new_b_demo",
  "hourly_rows": 1947,
  "vessels": 3,
  "alarm_2h_hours": 0,
  "low_support_rows": 0,
  "model_origin": "rebuilt_20261101_june_model",
  "as_of_causal": true,
  "original_final_test_b_replaced": false,
  "official_score": null,
  "target_start": "2026-06-01 00:00:00",
  "cdf_ties": "round12 midrank; thresholds recomputed on same rounded calibration",
  "past_target_rejected": true,
  "prefix_invariance": true,
  "prefix_score_max_error": 0.0
}



## 8. 실행 기록 저장
결과는 실행마다 새 폴더에 저장합니다. A의 예측 확률과 B의 상대 이상점수는 서로 의미가 다릅니다.
공식 정답과 채점 산식이 없으므로 과제별 공식 성적·합산 대회 점수는 비워 둡니다.
기존 제출 엑셀은 B가 9열에서 8,507열로 늘어났으므로 이 노트북은 수용 여부가 불명확한 엑셀을 새로 확정하지 않습니다.
대신 선박별 A CSV와 시간대별 B CSV를 저장합니다. 공식 제출 스키마가 확인되면 그 계약에 맞게 변환해야 합니다.


In [8]:
run_manifest = {
    'package_version': asset_manifest['version'], 'runtime_versions': versions,
    'assets_zip_sha256': file_sha(ASSET_ZIP), 'external_datasets_used': False,
    'task_a': a_summary, 'task_b_frozen': b_summary,
    'holdout_reexecuted': holdout_summary is not None, 'holdout': holdout_summary,
    'separate_rebuilt_b_inference': new_b_summary,
    'official_task_a_score': None, 'official_task_b_score': None,
    'official_combined_score': None, 'online_submission_performed': False
}
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('실행 완료:', OUTPUT_DIR)
if holdout_summary:
    print('로컬 선박 Macro F1:', f"{100*holdout_summary['vessel_aggregate_macro_f1']:.2f}%")
    print('로컬 선박 정확도:', f"{100*holdout_summary['vessel_aggregate_accuracy']:.2f}%")
print('과제 2 공식 성능 점수: 사고 정답이 없어 산출하지 않음')
print('생성 파일:', ', '.join(sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file() and not p.name.endswith(('.log', '_job.json')))))


실행 완료: /private/tmp/kisti_final_notebook_c12b003m/outputs/final_run_20261101_153354_473653
로컬 선박 Macro F1: 54.33%
로컬 선박 정확도: 56.87%
과제 2 공식 성능 점수: 사고 정답이 없어 산출하지 않음
생성 파일: evaluate_summary.json, frozen_b_summary.json, new_b_demo_summary.json, predict_a_summary.json, run_manifest.json, task_a_holdout_vessel_predictions.csv, task_a_hourly_probabilities.parquet, task_a_predictions.csv, task_b_frozen_hourly_scores.csv, task_b_frozen_peak_times.csv, task_b_rebuilt_demo_hourly_scores.csv, task_b_rebuilt_demo_peak_times.csv
